# LSTM Language Model for Fairy Tale Text Generation

A word-level LSTM language model trained on 170 fairy tale books from Project Gutenberg.
The model learns to predict the next token given a context window and can generate fairy-tale-style text.

## 1. Data Pipeline

### 1.1 Data Collection

We built a web crawler that downloads fairy tale texts from [Project Gutenberg](https://www.gutenberg.org/).
The crawler targets 7 bookshelves covering fairy tales, folklore, and children's literature:

- Applies a 2-second delay between HTTP requests (rate limiting / politeness)
- Filters for English-language books only
- Stores each book as a plain UTF-8 text file along with metadata (title, author, word count, SHA-256 hash)
- Idempotent: skips already-downloaded books on re-runs

The corpus was built incrementally: initial experiments (Notebooks 1–6) used **49 books** (~3.2M tokens), while later experiments (Notebooks 7+) expanded to **170 books** (~15.9M tokens).

### 1.2 Text Cleaning

Raw Project Gutenberg texts contain significant boilerplate — license headers, production credits, tables of contents, and footer disclaimers. A dedicated cleaning script removes these:

1. **Boilerplate stripping** — removes Project Gutenberg `*** START/END ***` markers and everything outside them
2. **Front matter removal** — strips production credits, illustration markers, and editorial notes in the first ~400 lines
3. **Table of contents removal** — detects `CONTENTS` headers and removes the block until the first real chapter heading
4. **Story start detection** — locates the actual narrative beginning via patterns like "Chapter I", "Preface", or Roman numeral headings
5. **Whitespace normalization** — collapses runs of 4+ blank lines down to 3

### 1.3 Tokenization & Vocabulary Building

The cleaned texts are tokenized into a fixed vocabulary suitable for neural network training:

| Step | Description |
|------|-------------|
| Lowercasing | All text converted to lowercase |
| Number replacement | All digits replaced with a `<num>` token |
| Quote/dash normalization | Curly quotes → straight quotes, em-dashes → `—` |
| Word extraction | Regex `[a-z]+(?:'[a-z]+)?` captures words and contractions |
| Punctuation tokens | `.,!?;:\-—()[]"'` kept as individual tokens |
| Vocabulary cap | Top 30,000 most frequent tokens (min frequency ≥ 2) |
| Special tokens | `<pad>`, `<unk>`, `<bos>`, `<eos>`, `<num>` |

Each book is then serialized as a sequence of integer token IDs and saved to disk.

**Final corpus (170 books)**: ~15.9 million tokens, vocabulary of 30,000 tokens.

## 2. Configuration & Imports

All hyperparameters are defined upfront. The key settings for the final model:
- **Sequence length**: 70 tokens (context window for next-token prediction)
- **Stride**: 10 tokens between windows (overlapping samples for more training data)
- **Batch size**: 128
- **Per-book split**: 80% train / 10% validation / 10% test (contiguous segments within each book)

In [1]:
import os
import csv
import math
import random
import json
import glob
from datetime import datetime
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata_clean.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 70        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 128
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

# Per-book split ratios (each book is split into 3 contiguous parts)
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000

# ---------- Checkpoint config ----------
CKPT_DIR = 'checkpoints'
SAVE_EVERY_STEPS = 100
PRUNE_VAL_LOSS_THRESHOLD = 0.1

RESUME_FROM = None
# ------------------------------------

use_amp = (DEVICE == "cuda")

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## 3. Load Vocabulary & Metadata

Load the pre-built vocabulary (30,000 tokens) and book metadata from the preprocessing pipeline.

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 30000
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 170


## 4. Dataset: Windowed Next-Token Prediction

The language modeling objective is **next-token prediction**: given a sequence of tokens $[t_1, t_2, \ldots, t_n]$, predict $[t_2, t_3, \ldots, t_{n+1}]$ at each position.

Training samples are created using a **sliding window** approach:
- A window of `SEQ_LEN + 1` tokens is extracted from each book
- The first `SEQ_LEN` tokens are the input (x), the last `SEQ_LEN` tokens are the target (y)
- Windows overlap with a stride of 10 tokens, creating dense coverage of the text

**Data split strategy**: Each book is split into three contiguous segments (80/10/10) rather than random sampling. This prevents the model from seeing future context of a passage during training — it can only learn from the first 80% of each book, and is evaluated on unseen continuations.

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_book_tokens(ids: List[int], train_r: float, val_r: float) -> Tuple[List[int], List[int], List[int]]:
    n = len(ids)
    train_end = int(n * train_r)
    val_end = int(n * (train_r + val_r))
    return ids[:train_end], ids[train_end:val_end], ids[val_end:]

class CachedWindowDataset(Dataset):
    def __init__(self, token_chunks: List[torch.Tensor], seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens = []
        self.samples: List[Tuple[int, int]] = []

        for chunk in token_chunks:
            if chunk.numel() < seq_len + 2:
                continue
            bi = len(self.book_tokens)
            self.book_tokens.append(chunk)
            max_start = chunk.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)
        return chunk[:-1], chunk[1:]

# --- Collect books and split each book's tokens ---
books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_chunks = []
val_chunks = []
test_chunks = []

total_train_tokens = 0
total_val_tokens = 0
total_test_tokens = 0

for b in books:
    ids = load_ids_file(b.ids_path)
    tr, va, te = split_book_tokens(ids, TRAIN_RATIO, VAL_RATIO)
    train_chunks.append(torch.tensor(tr, dtype=torch.int32))
    val_chunks.append(torch.tensor(va, dtype=torch.int32))
    test_chunks.append(torch.tensor(te, dtype=torch.int32))
    total_train_tokens += len(tr)
    total_val_tokens += len(va)
    total_test_tokens += len(te)

train_ds = CachedWindowDataset(train_chunks, SEQ_LEN, STRIDE)
val_ds   = CachedWindowDataset(val_chunks, SEQ_LEN, STRIDE)
test_ds  = CachedWindowDataset(test_chunks, SEQ_LEN, STRIDE)

print(f'Books: {len(books)} (each split {TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%})')
print(f'Tokens: train={total_train_tokens:,}  val={total_val_tokens:,}  test={total_test_tokens:,}')
print(f'Samples: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: 170 (each split 80%/10%/10%)
Tokens: train=12,734,962  val=1,591,864  test=1,591,961
Samples: train=1,272,381  val=158,081  test=158,089


## 5. Model Architecture: Stacked LSTM with Weight Tying

The model is a classic LSTM language model with three components:

1. **Embedding layer** — maps each token ID to a dense vector of `emb_dim` dimensions
2. **Stacked LSTM** — processes the sequence through multiple LSTM layers with inter-layer dropout
3. **Output projection** — a linear layer that maps the LSTM's hidden state back to vocabulary-sized logits

**Weight tying**: The output projection matrix shares its weights with the embedding matrix (`fc.weight = emb.weight`). This is a key regularization technique that:
- Reduces the total parameter count (the embedding matrix is large: `vocab_size × emb_dim`)
- Forces the model to learn embeddings that are useful for both input representation and output prediction
- Has been shown to consistently improve language model perplexity ([Press & Wolf, 2017](https://arxiv.org/abs/1608.05859))

When weight tying is used, the embedding dimension directly determines the output projection size, which is a constraint on architecture design.

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, num_layers, dropout, pad_id):
        super().__init__()
        # For Weight Tying, emb_dim must usually equal hidden
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.fc = nn.Linear(hidden, vocab_size)

        # --- WEIGHT TYING ---
        # This ties the input and output weights
        self.fc.weight = self.emb.weight

    def forward(self, x):
        e = self.emb(x)
        # Note: Standard nn.LSTM already applies dropout BETWEEN layers
        out, _ = self.lstm(e)
        # Final linear layer uses tied weights
        logits = self.fc(out)
        return logits

## 6. Text Generation (Sampling)

Given a prompt, the model generates text autoregressively — predicting one token at a time and feeding it back as input. We use **top-k sampling** with temperature scaling to control the diversity of generated text.

In [5]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

## 7. Evaluation Metrics

We evaluate the model using five complementary metrics:

| Metric | Formula | Interpretation | Range |
|--------|---------|---------------|-------|
| **Cross-Entropy Loss** | $-\frac{1}{N}\sum_{i} \log p(t_i \mid t_{<i})$ | Average negative log-likelihood per token | [0, ∞) — lower is better |
| **Perplexity (PPL)** | $e^{\text{CE Loss}}$ | "Effective vocabulary size" the model is choosing from | [1, ∞) — lower is better |
| **Top-1 Accuracy** | $\frac{1}{N}\sum_i \mathbb{1}[\hat{t}_i = t_i]$ | Fraction where the #1 prediction is correct | [0, 1] — higher is better |
| **Top-5 Accuracy** | $\frac{1}{N}\sum_i \mathbb{1}[t_i \in \text{top5}(\hat{t}_i)]$ | Fraction where the correct token is in the top 5 | [0, 1] — higher is better |
| **MRR** | $\frac{1}{N}\sum_i \frac{1}{\text{rank}(t_i)}$ | Average inverse rank of the correct token | [0, 1] — higher is better |

**Perplexity** is the primary metric for language models. For our 30,000-token vocabulary:
- Random guessing → PPL ≈ 30,000
- A model with PPL = 70 is as uncertain as choosing among ~70 equally likely tokens at each step
- State-of-the-art neural LMs on large English corpora achieve PPL < 20, but our domain-specific corpus and model size make PPL 60–70 a strong result

**MRR** (Mean Reciprocal Rank) complements accuracy by capturing *how close* the model is even when the top prediction is wrong — a rank-2 prediction (MRR contribution = 0.5) is much better than rank-1000 (MRR ≈ 0).

In [6]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

def run_eval(model, loader=None) -> dict:
    """Evaluate model, return dict with loss, ppl, top1_acc, top5_acc, mrr."""
    if loader is None:
        loader = val_loader
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    total_top1 = 0
    total_top5 = 0
    total_rr = 0.0  # sum of reciprocal ranks

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

            # Flatten for metric computation
            flat_logits = logits.reshape(-1, logits.size(-1))  # (B*T, V)
            flat_y = y.reshape(-1)                              # (B*T,)
            flat_mask = mask.reshape(-1)                        # (B*T,)

            # Top-1 accuracy
            pred_top1 = flat_logits.argmax(dim=-1)
            total_top1 += int(((pred_top1 == flat_y) & flat_mask).sum().item())

            # Top-5 accuracy
            _, pred_top5 = flat_logits.topk(5, dim=-1)  # (B*T, 5)
            target_exp = flat_y.unsqueeze(-1).expand_as(pred_top5)
            hits5 = (pred_top5 == target_exp).any(dim=-1) & flat_mask
            total_top5 += int(hits5.sum().item())

            # MRR (mean reciprocal rank)
            target_logits = flat_logits.gather(1, flat_y.unsqueeze(-1))  # (B*T, 1)
            ranks = (flat_logits > target_logits).sum(dim=-1).float()    # 0-indexed
            rr = 1.0 / (ranks + 1.0)
            total_rr += float((rr * flat_mask.float()).sum().item())

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    top1_acc = total_top1 / max(1, total_tokens)
    top5_acc = total_top5 / max(1, total_tokens)
    mrr = total_rr / max(1, total_tokens)
    return {
        'loss': avg_loss,
        'ppl': ppl,
        'top1_acc': top1_acc,
        'top5_acc': top5_acc,
        'mrr': mrr,
    }

## 8. Training

**Optimization strategy**:
- **Optimizer**: AdamW (Adam with decoupled weight decay of 0.01)
- **Learning rate schedule**: ReduceLROnPlateau — halves LR after 1 epoch without validation improvement
- **Early stopping**: Stops training after 4 consecutive epochs with no validation loss improvement
- **Gradient clipping**: Max norm of 1.0 to prevent exploding gradients (a common issue with LSTMs)
- **Mixed precision**: FP16 forward/backward pass with automatic loss scaling (GradScaler) for ~2× GPU speedup

**Grid search**: We explore combinations of learning rate, hidden size, dropout, and number of layers. Each configuration is trained independently with early stopping, and the best model (by validation loss) is selected for final test evaluation.

In [7]:
def make_run_dir(base_dir: str) -> str:
    """Create a new run subfolder named YYYYMMDDHHMMSS."""
    name = datetime.now().strftime('%Y%m%d%H%M%S')
    path = os.path.join(base_dir, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_hyperparams_dict(cfg: dict) -> dict:
    """Collect all hyperparameters needed to reproduce a run."""
    return {
        'seed': SEED,
        'seq_len': SEARCH_SEQ_LEN,
        'stride': SEARCH_STRIDE,
        'batch_size': SEARCH_BATCH_SIZE,
        'grad_clip': GRAD_CLIP,
        'emb_dim': SEARCH_EMB_DIM,
        'num_layers': cfg['num_layers'],
        'vocab_size': len(vocab),
        'lr': cfg['lr'],
        'hidden_size': cfg['hidden_size'],
        'dropout': cfg['dropout'],
        'steps_per_epoch': SEARCH_STEPS_PER_EPOCH,
        'max_epochs': SEARCH_EPOCHS,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'sched_factor': SCHED_FACTOR,
        'sched_patience': SCHED_PATIENCE,
        'sched_min_lr': SCHED_MIN_LR,
        'train_ratio': TRAIN_RATIO,
        'val_ratio': VAL_RATIO,
        'test_ratio': TEST_RATIO,
    }

def save_checkpoint(
    run_dir: str,
    filename: str,
    model: nn.Module,
    optimizer,
    scaler,
    scheduler,
    epoch: int,
    global_step: int,
    train_loss: float,
    val_loss: Optional[float],
    val_ppl: Optional[float],
    hyperparams: dict,
    is_step_ckpt: bool,
):
    path = os.path.join(run_dir, filename)
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'global_step': global_step,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_ppl': val_ppl,
        'hyperparams': hyperparams,
        'is_step_ckpt': is_step_ckpt,
    }
    torch.save(ckpt, path)
    return path

def prune_checkpoints(run_dir: str, threshold: float):
    """Delete epoch checkpoints whose val_loss exceeds best_val_loss + threshold.
    Step checkpoints are always deleted (they are crash recovery only)."""
    ckpt_files = sorted(glob.glob(os.path.join(run_dir, '*.pt')))
    if not ckpt_files:
        return

    epoch_ckpts = []  # (path, val_loss)
    for f in ckpt_files:
        ckpt = torch.load(f, map_location='cpu', weights_only=False)
        if ckpt.get('is_step_ckpt', False):
            os.remove(f)
            print(f"    pruned step ckpt: {os.path.basename(f)}")
        else:
            val_loss = ckpt.get('val_loss')
            if val_loss is not None:
                epoch_ckpts.append((f, val_loss))

    if not epoch_ckpts:
        return

    best_val = min(vl for _, vl in epoch_ckpts)
    for f, vl in epoch_ckpts:
        if vl > best_val + threshold:
            os.remove(f)
            print(f"    pruned epoch ckpt: {os.path.basename(f)} (val_loss={vl:.4f}, best={best_val:.4f}, threshold={threshold})")

def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

print("Checkpoint helpers ready.")

Checkpoint helpers ready.


In [8]:
import itertools

# ---------- Search space ----------
SEARCH_GRID = {
    'lr':          [0.001],
    'hidden_size': [1024],
    'dropout':     [0.4],
    'num_layers':  [3],
}

SEARCH_EMB_DIM    = 1024
SEARCH_EPOCHS     = 500
SEARCH_STEPS_PER_EPOCH = 50000
SEARCH_BATCH_SIZE = 128
SEARCH_SEQ_LEN    = 70
SEARCH_STRIDE     = 10
EARLY_STOP_PATIENCE = 4

# ReduceLROnPlateau settings
SCHED_FACTOR  = 0.5            # halve LR when plateau
SCHED_PATIENCE = 1             # reduce after 1 epoch of no improvement
SCHED_MIN_LR  = 1e-5           # don't go below this

# Build all configs
keys = list(SEARCH_GRID.keys())
combos = list(itertools.product(*[SEARCH_GRID[k] for k in keys]))
configs = [dict(zip(keys, c)) for c in combos]

print(f"Total configurations: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"  [{i+1}] lr={cfg['lr']}, hidden={cfg['hidden_size']}, dropout={cfg['dropout']}, layers={cfg['num_layers']}")

Total configurations: 1
  [1] lr=0.001, hidden=1024, dropout=0.4, layers=3


In [9]:
def run_search_eval(mdl, loader=None) -> dict:
    """Evaluate model on a given loader, return dict with loss, ppl, top1_acc, top5_acc, mrr."""
    if loader is None:
        loader = val_loader
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    total_top1 = 0
    total_top5 = 0
    total_rr = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_y = y.reshape(-1)
            flat_mask = mask.reshape(-1)

            # Top-1 accuracy
            pred_top1 = flat_logits.argmax(dim=-1)
            total_top1 += int(((pred_top1 == flat_y) & flat_mask).sum().item())

            # Top-5 accuracy
            _, pred_top5 = flat_logits.topk(5, dim=-1)
            target_exp = flat_y.unsqueeze(-1).expand_as(pred_top5)
            hits5 = (pred_top5 == target_exp).any(dim=-1) & flat_mask
            total_top5 += int(hits5.sum().item())

            # MRR (mean reciprocal rank)
            target_logits = flat_logits.gather(1, flat_y.unsqueeze(-1))
            ranks = (flat_logits > target_logits).sum(dim=-1).float()
            rr = 1.0 / (ranks + 1.0)
            total_rr += float((rr * flat_mask.float()).sum().item())

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    top1_acc = total_top1 / max(1, total_tokens)
    top5_acc = total_top5 / max(1, total_tokens)
    mrr = total_rr / max(1, total_tokens)
    return {'loss': avg_loss, 'ppl': ppl, 'top1_acc': top1_acc, 'top5_acc': top5_acc, 'mrr': mrr}


# ---------- Resume handling ----------
resume_ckpt = None
if RESUME_FROM is not None:
    if not os.path.exists(RESUME_FROM):
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_FROM}")
    resume_ckpt = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
    resume_run_dir = os.path.dirname(RESUME_FROM)
    rh = resume_ckpt['hyperparams']
    configs = [{
        'lr': rh['lr'],
        'hidden_size': rh['hidden_size'],
        'dropout': rh['dropout'],
        'num_layers': rh['num_layers'],
    }]
    SEARCH_EMB_DIM = rh['emb_dim']
    print(f"Resuming from: {RESUME_FROM}")
    print(f"  epoch={resume_ckpt['epoch']}, global_step={resume_ckpt['global_step']}, "
          f"train_loss={resume_ckpt['train_loss']:.4f}")
    if resume_ckpt.get('val_loss') is not None:
        print(f"  val_loss={resume_ckpt['val_loss']:.4f}, val_ppl={resume_ckpt['val_ppl']:.2f}")

search_results = []
best_model_state = None

actual_steps = min(SEARCH_STEPS_PER_EPOCH, len(train_loader))

for cfg_idx, cfg in enumerate(configs):
    print(f"\n{'='*70}")
    print(f"Config [{cfg_idx+1}/{len(configs)}]  lr={cfg['lr']}  hidden={cfg['hidden_size']}  "
          f"dropout={cfg['dropout']}  layers={cfg['num_layers']}")
    print(f"Steps per epoch: {actual_steps} (loader has {len(train_loader)} batches, cap {SEARCH_STEPS_PER_EPOCH})")
    print('='*70)

    set_seed(SEED)
    hparams = build_hyperparams_dict(cfg)

    # Create or reuse run directory
    if resume_ckpt is not None and cfg_idx == 0:
        run_dir = resume_run_dir
    else:
        run_dir = make_run_dir(CKPT_DIR)
    print(f"Run dir: {run_dir}")

    with open(os.path.join(run_dir, 'hyperparams.json'), 'w') as f:
        json.dump(hparams, f, indent=2)

    mdl = LSTMLM(
        vocab_size=len(vocab),
        emb_dim=SEARCH_EMB_DIM,
        hidden=cfg['hidden_size'],
        num_layers=cfg['num_layers'],
        dropout=cfg['dropout'],
        pad_id=pad_id,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in mdl.parameters())
    print(f"Model params: {n_params:,}")

    crit = nn.CrossEntropyLoss(ignore_index=pad_id)
    opt  = torch.optim.AdamW(mdl.parameters(), lr=cfg['lr'], weight_decay=0.01)
    scl  = GradScaler("cuda", enabled=use_amp)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=SCHED_FACTOR,
        patience=SCHED_PATIENCE, min_lr=SCHED_MIN_LR
    )

    start_epoch = 1
    global_step = 0
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0
    best_state = None
    best_metrics = None

    if resume_ckpt is not None and cfg_idx == 0:
        mdl.load_state_dict(resume_ckpt['model_state_dict'])
        opt.load_state_dict(resume_ckpt['optimizer_state_dict'])
        scl.load_state_dict(resume_ckpt['scaler_state_dict'])
        scheduler.load_state_dict(resume_ckpt['scheduler_state_dict'])
        global_step = resume_ckpt['global_step']

        if resume_ckpt.get('is_step_ckpt', False):
            start_epoch = resume_ckpt['epoch']
            print(f"  Resuming mid-epoch: restarting epoch {start_epoch} from step 0")
        else:
            start_epoch = resume_ckpt['epoch'] + 1
            print(f"  Resuming after epoch {resume_ckpt['epoch']}: starting epoch {start_epoch}")

        for f in sorted(glob.glob(os.path.join(run_dir, 'epoch_*.pt'))):
            c = torch.load(f, map_location='cpu', weights_only=False)
            vl = c.get('val_loss')
            if vl is not None and vl < best_val_loss:
                best_val_loss = vl
                best_epoch = c['epoch']
                best_state = c['model_state_dict']
        if best_val_loss < float('inf'):
            print(f"  Restored best_val_loss={best_val_loss:.4f} from epoch {best_epoch}")

        resume_ckpt = None

    for epoch in range(start_epoch, SEARCH_EPOCHS + 1):
        mdl.train()
        running = 0.0
        seen_tokens = 0

        pbar = tqdm(enumerate(train_loader, start=1),
                     total=actual_steps,
                     desc=f"  E{epoch}/{SEARCH_EPOCHS}", leave=False)

        for step, (x, y) in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
            scl.step(opt)
            scl.update()

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            running += float(loss.item()) * tok
            seen_tokens += tok
            global_step += 1

            avg = running / max(1, seen_tokens)
            pbar.set_postfix(loss=f"{avg:.4f}", step=global_step)

            if SAVE_EVERY_STEPS > 0 and step % SAVE_EVERY_STEPS == 0:
                step_fname = f"step_{global_step:07d}.pt"
                save_checkpoint(
                    run_dir, step_fname, mdl, opt, scl, scheduler,
                    epoch=epoch, global_step=global_step,
                    train_loss=avg, val_loss=None, val_ppl=None,
                    hyperparams=hparams, is_step_ckpt=True,
                )

            if step >= actual_steps:
                break
        pbar.close()

        # --- Epoch-end: evaluate with all metrics ---
        val_m = run_search_eval(mdl)
        train_loss = running / max(1, seen_tokens)
        current_lr = opt.param_groups[0]['lr']
        print(f"  epoch {epoch}  train_loss={train_loss:.4f}  "
              f"val_loss={val_m['loss']:.4f}  val_ppl={val_m['ppl']:.2f}  "
              f"top1={val_m['top1_acc']:.4f}  top5={val_m['top5_acc']:.4f}  "
              f"mrr={val_m['mrr']:.4f}  lr={current_lr:.6f}", end="")

        epoch_fname = f"epoch_{epoch:03d}.pt"
        save_checkpoint(
            run_dir, epoch_fname, mdl, opt, scl, scheduler,
            epoch=epoch, global_step=global_step,
            train_loss=train_loss, val_loss=val_m['loss'], val_ppl=val_m['ppl'],
            hyperparams=hparams, is_step_ckpt=False,
        )

        scheduler.step(val_m['loss'])

        if val_m['loss'] < best_val_loss:
            best_val_loss = val_m['loss']
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in mdl.state_dict().items()}
            best_metrics = val_m.copy()
            print(f"  *best* [saved {epoch_fname}]")
        else:
            patience_counter += 1
            print(f"  (no improve {patience_counter}/{EARLY_STOP_PATIENCE}) [saved {epoch_fname}]")
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  >> early stop at epoch {epoch}, best was epoch {best_epoch}")
                break

        print(f"  Pruning checkpoints (threshold={PRUNE_VAL_LOSS_THRESHOLD})...")
        prune_checkpoints(run_dir, PRUNE_VAL_LOSS_THRESHOLD)

    search_results.append({
        **cfg,
        'best_val_loss': best_val_loss,
        'best_val_ppl': math.exp(min(20.0, best_val_loss)),
        'best_top1': best_metrics['top1_acc'] if best_metrics else 0.0,
        'best_top5': best_metrics['top5_acc'] if best_metrics else 0.0,
        'best_mrr': best_metrics['mrr'] if best_metrics else 0.0,
        'best_epoch': best_epoch,
        'total_epochs': epoch,
        'best_state': best_state,
        'run_dir': run_dir,
    })

    del mdl, opt, scl, crit, scheduler
    torch.cuda.empty_cache()

print("\n\nGrid search complete.")


Config [1/1]  lr=0.001  hidden=1024  dropout=0.4  layers=3
Steps per epoch: 9940 (loader has 9940 batches, cap 50000)
Run dir: checkpoints\20260329111509
Model params: 55,940,400


  E1/500:   0%|          | 0/9940 [00:00<?, ?it/s]

  epoch 1  train_loss=4.3393  val_loss=4.1883  val_ppl=65.91  top1=0.2720  top5=0.5061  mrr=0.3820  lr=0.001000  *best* [saved epoch_001.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0000100.pt
    pruned step ckpt: step_0000200.pt
    pruned step ckpt: step_0000300.pt
    pruned step ckpt: step_0000400.pt
    pruned step ckpt: step_0000500.pt
    pruned step ckpt: step_0000600.pt
    pruned step ckpt: step_0000700.pt
    pruned step ckpt: step_0000800.pt
    pruned step ckpt: step_0000900.pt
    pruned step ckpt: step_0001000.pt
    pruned step ckpt: step_0001100.pt
    pruned step ckpt: step_0001200.pt
    pruned step ckpt: step_0001300.pt
    pruned step ckpt: step_0001400.pt
    pruned step ckpt: step_0001500.pt
    pruned step ckpt: step_0001600.pt
    pruned step ckpt: step_0001700.pt
    pruned step ckpt: step_0001800.pt
    pruned step ckpt: step_0001900.pt
    pruned step ckpt: step_0002000.pt
    pruned step ckpt: step_0002100.pt
    pruned step ckpt

  E2/500:   0%|          | 0/9940 [00:00<?, ?it/s]

  epoch 2  train_loss=3.8976  val_loss=4.1172  val_ppl=61.39  top1=0.2782  top5=0.5142  mrr=0.3890  lr=0.001000  *best* [saved epoch_002.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0010040.pt
    pruned step ckpt: step_0010140.pt
    pruned step ckpt: step_0010240.pt
    pruned step ckpt: step_0010340.pt
    pruned step ckpt: step_0010440.pt
    pruned step ckpt: step_0010540.pt
    pruned step ckpt: step_0010640.pt
    pruned step ckpt: step_0010740.pt
    pruned step ckpt: step_0010840.pt
    pruned step ckpt: step_0010940.pt
    pruned step ckpt: step_0011040.pt
    pruned step ckpt: step_0011140.pt
    pruned step ckpt: step_0011240.pt
    pruned step ckpt: step_0011340.pt
    pruned step ckpt: step_0011440.pt
    pruned step ckpt: step_0011540.pt
    pruned step ckpt: step_0011640.pt
    pruned step ckpt: step_0011740.pt
    pruned step ckpt: step_0011840.pt
    pruned step ckpt: step_0011940.pt
    pruned step ckpt: step_0012040.pt
    pruned step ckpt

...

  epoch 5  train_loss=3.4903  val_loss=4.1385  val_ppl=62.71  top1=0.2796  top5=0.5158  mrr=0.3905  lr=0.001000  (no improve 2/4) [saved epoch_005.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0039860.pt
    pruned step ckpt: step_0039960.pt
    pruned step ckpt: step_0040060.pt
    pruned step ckpt: step_0040160.pt
    pruned step ckpt: step_0040260.pt
    pruned step ckpt: step_0040360.pt
    pruned step ckpt: step_0040460.pt
    pruned step ckpt: step_0040560.pt
    pruned step ckpt: step_0040660.pt
    pruned step ckpt: step_0040760.pt
    pruned step ckpt: step_0040860.pt
    pruned step ckpt: step_0040960.pt
    pruned step ckpt: step_0041060.pt
    pruned step ckpt: step_0041160.pt
    pruned step ckpt: step_0041260.pt
    pruned step ckpt: step_0041360.pt
    pruned step ckpt: step_0041460.pt
    pruned step ckpt: step_0041560.pt
    pruned step ckpt: step_0041660.pt
    pruned step ckpt: step_0041760.pt
    pruned step ckpt: step_0041860.pt
    pruned

  E6/500:   0%|          | 0/9940 [00:00<?, ?it/s]

  epoch 6  train_loss=3.3114  val_loss=4.1747  val_ppl=65.02  top1=0.2793  top5=0.5151  mrr=0.3899  lr=0.000500  (no improve 3/4) [saved epoch_006.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0049800.pt
    pruned step ckpt: step_0049900.pt
    pruned step ckpt: step_0050000.pt
    pruned step ckpt: step_0050100.pt
    pruned step ckpt: step_0050200.pt
    pruned step ckpt: step_0050300.pt
    pruned step ckpt: step_0050400.pt
    pruned step ckpt: step_0050500.pt
    pruned step ckpt: step_0050600.pt
    pruned step ckpt: step_0050700.pt
    pruned step ckpt: step_0050800.pt
    pruned step ckpt: step_0050900.pt
    pruned step ckpt: step_0051000.pt
    pruned step ckpt: step_0051100.pt
    pruned step ckpt: step_0051200.pt
    pruned step ckpt: step_0051300.pt
    pruned step ckpt: step_0051400.pt
    pruned step ckpt: step_0051500.pt
    pruned step ckpt: step_0051600.pt
    pruned step ckpt: step_0051700.pt
    pruned step ckpt: step_0051800.pt
    pruned

  E7/500:   0%|          | 0/9940 [00:00<?, ?it/s]

  epoch 7  train_loss=3.2277  val_loss=4.2117  val_ppl=67.47  top1=0.2781  top5=0.5130  mrr=0.3883  lr=0.000500  (no improve 4/4) [saved epoch_007.pt]
  >> early stop at epoch 7, best was epoch 3


Grid search complete.


## 9. Results & Test Evaluation

Display the grid search results ranked by validation loss, then evaluate the best model on the held-out test set.

In [10]:
# Sort by best validation loss and display results
search_results.sort(key=lambda r: r['best_val_loss'])

print(f"{'Rank':<5} {'LR':<10} {'Hidden':<8} {'Layers':<8} {'Drop':<7} {'Val Loss':<10} {'Val PPL':<10} "
      f"{'Top1':<8} {'Top5':<8} {'MRR':<8} {'Best@':<7} {'Stopped':<7} {'Run Dir'}")
print('-' * 120)
for i, r in enumerate(search_results):
    print(f"{i+1:<5} {r['lr']:<10} {r['hidden_size']:<8} {r['num_layers']:<8} {r['dropout']:<7.2f} "
          f"{r['best_val_loss']:<10.4f} {r['best_val_ppl']:<10.2f} "
          f"{r['best_top1']:<8.4f} {r['best_top5']:<8.4f} {r['best_mrr']:<8.4f} "
          f"{r['best_epoch']:<7} {r['total_epochs']:<7} {r['run_dir']}")

best = search_results[0]
print(f"\nBest config:")
print(f"  lr={best['lr']}, hidden_size={best['hidden_size']}, num_layers={best['num_layers']}, dropout={best['dropout']}")
print(f"  val_loss={best['best_val_loss']:.4f}, val_ppl={best['best_val_ppl']:.2f}, "
      f"top1={best['best_top1']:.4f}, top5={best['best_top5']:.4f}, mrr={best['best_mrr']:.4f}")
print(f"  best at epoch {best['best_epoch']}, run_dir={best['run_dir']}")

# --- Final test evaluation (only done once, on best model) ---
print(f"\n{'='*60}")
print("Final TEST evaluation (best model from grid search)")
print('='*60)

best_mdl = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=SEARCH_EMB_DIM,
    hidden=best['hidden_size'],
    num_layers=best['num_layers'],
    dropout=best['dropout'],
    pad_id=pad_id,
).to(DEVICE)
best_mdl.load_state_dict(best['best_state'])

test_m = run_search_eval(best_mdl, loader=test_loader)
print(f"  test_loss={test_m['loss']:.4f}  test_ppl={test_m['ppl']:.2f}  "
      f"top1={test_m['top1_acc']:.4f}  top5={test_m['top5_acc']:.4f}  mrr={test_m['mrr']:.4f}")

# List remaining checkpoints for best run
print(f"\nCheckpoints in {best['run_dir']}:")
for f in sorted(glob.glob(os.path.join(best['run_dir'], '*.pt'))):
    print(f"  {os.path.basename(f)}")

# Clean up saved states from search_results to free memory
for r in search_results:
    r.pop('best_state', None)
del best_mdl
torch.cuda.empty_cache()

Rank  LR         Hidden   Layers   Drop    Val Loss   Val PPL    Top1     Top5     MRR      Best@   Stopped Run Dir
------------------------------------------------------------------------------------------------------------------------
1     0.001      1024     3        0.40    4.1040     60.58      0.2802   0.5173   0.3913   3       7       checkpoints\20260329111509

Best config:
  lr=0.001, hidden_size=1024, num_layers=3, dropout=0.4
  val_loss=4.1040, val_ppl=60.58, top1=0.2802, top5=0.5173, mrr=0.3913
  best at epoch 3, run_dir=checkpoints\20260329111509

Final TEST evaluation (best model from grid search)
  test_loss=4.2167  test_ppl=67.81  top1=0.2819  top5=0.5125  mrr=0.3901

Checkpoints in checkpoints\20260329111509:
  epoch_001.pt
  epoch_002.pt
  epoch_003.pt
  epoch_004.pt
  epoch_005.pt
  epoch_006.pt
  epoch_007.pt
  step_0059740.pt
  step_0059840.pt
  step_0059940.pt
  step_0060040.pt
  step_0060140.pt
  step_0060240.pt
  step_0060340.pt
  step_0060440.pt
  step_0060540

## 10. Grid Search Results: All Experiments

Over the course of development, we ran extensive grid searches across multiple notebooks, progressively refining the architecture and hyperparameters.

### Phase 1: Width Scaling (2-layer LSTM, 49 books, 90/10 book-level split)

Notebooks 1–4 used **49 books** (~3.2M tokens) with a 90/10 split by whole books (44 train, 5 val — no test set). This phase focused on finding the optimal hidden size and learning rate for a 2-layer LSTM.

**Notebook 1 — Grid Search A** (27 configs): EMB=256, Layers=2, SEQ=50, Steps=1,500/epoch, 8 epochs

| Rank | LR | Hidden | Dropout | Val PPL |
|------|-----|--------|---------|---------|
| 1 | 0.002 | 768 | 0.5 | 70.08 |
| 2 | 0.001 | 768 | 0.3 | 71.01 |
| 3 | 0.001 | 768 | 0.5 | 71.21 |
| ... | ... | 256 | ... | 80–120 |

**Notebook 1 — Grid Search B** (18 configs): same setup, wider hidden range

| Rank | LR | Hidden | Dropout | Val PPL |
|------|-----|--------|---------|---------|
| 1 | 0.002 | 1024 | 0.5 | 68.34 |
| 2 | 0.0015 | 1024 | 0.5 | 68.87 |
| 3 | 0.002 | 1024 | 0.4 | 68.92 |
| ... | ... | 768 | ... | 70–72 |

**Notebook 2** (2 configs): EMB=256, Layers=2, Hidden=2048

| Rank | LR | Hidden | Dropout | Val PPL |
|------|-----|--------|---------|---------|
| 1 | 0.0015 | 2048 | 0.5 | 63.70 |
| 2 | 0.002 | 2048 | 0.5 | 130.93 |

**Notebook 3** (4 configs): EMB=256, Layers=2, Hidden=2048, 15 epochs, early stopping (patience=3)

| Rank | LR | Hidden | Dropout | Val PPL | Best Epoch |
|------|-----|--------|---------|---------|------------|
| 1 | 0.001 | 2048 | 0.6 | 64.86 | 3 |
| 2 | 0.001 | 2048 | 0.5 | 66.10 | 3 |
| 3 | 0.0005 | 2048 | 0.6 | 67.99 | 4 |
| 4 | 0.0005 | 2048 | 0.5 | 68.26 | 4 |

**Notebook 4** (1 config): Low LR with ReduceLROnPlateau scheduler

| LR | Hidden | Dropout | Val PPL | Best Epoch |
|-----|--------|---------|---------|------------|
| 0.00025 | 2048 | 0.6 | 72.62 | 5 |

### Phase 1b: Split Strategy Change (49 books, per-book 80/10/10 split)

Starting from Notebook 5, the data split changed from whole-book to per-book (80% train / 10% val / 10% test within each book). Still using 49 books.

**Notebook 5** (1 config): Testing extreme width — EMB=512, Layers=2

| LR | EMB | Hidden | Dropout | Val PPL | Notes |
|-----|-----|--------|---------|---------|-------|
| 0.0005 | 512 | 4096 | 0.6 | 95.66 | Early stopped after epoch 1 — immediate overfitting |

**Notebook 6** (6 configs): EMB=256, Layers=2, Hidden=2048, 25 epochs

| Rank | LR | Hidden | Dropout | Val PPL | Best Epoch |
|------|-----|--------|---------|---------|------------|
| 1 | 0.00075 | 2048 | 0.5 | 96.66 | 2 |
| 2 | 0.00075 | 2048 | 0.6 | 97.01 | 3 |
| 3 | 0.0005 | 2048 | 0.6 | 99.33 | 4 |
| 4 | 0.00075 | 2048 | 0.55 | 99.45 | 3 |
| 5 | 0.0005 | 2048 | 0.5 | 101.29 | 3 |
| 6 | 0.0005 | 2048 | 0.55 | 101.71 | 4 |

> PPL is higher than Phase 1 because the per-book split is a harder evaluation — the model must predict continuations of each book rather than being tested on entirely separate (potentially easier) books.

### Phase 2: Expanded Corpus & Depth Exploration (170 books, per-book 80/10/10 split)

Starting from Notebook 7, the corpus was expanded from 49 to **170 books** (~15.9M tokens). Top-1, Top-5, and MRR metrics were also introduced.

> **Note**: Results from Phase 2 are not directly comparable to Phase 1 due to both the different split strategy and the larger corpus.

**Notebook 7** (16 configs): EMB=256, SEQ=50, Hidden ∈ {1024, 1536}, Layers ∈ {2, 3}, Dropout ∈ {0.5, 0.6}

| Rank | LR | Hidden | Layers | Drop | Val PPL | Top-1 | Top-5 | MRR |
|------|-----|--------|--------|------|---------|-------|-------|-----|
| 1 | 0.00075 | 1536 | 2 | 0.6 | 101.41 | 0.239 | 0.467 | 0.347 |
| 2 | 0.0005 | 1536 | 2 | 0.6 | 101.51 | 0.240 | 0.469 | 0.349 |
| 3 | 0.00075 | 1024 | 2 | 0.6 | 102.11 | 0.237 | 0.464 | 0.345 |
| 8 | 0.00075 | 1536 | 3 | 0.6 | 104.61 | 0.237 | 0.464 | 0.345 |
| 16 | 0.0005 | 1024 | 3 | 0.6 | 114.45 | 0.226 | 0.445 | 0.331 |

*Test evaluation (best model)*: PPL=127.05, Top-1=0.235, Top-5=0.458

**Notebook 7.1** (3 configs): Deep narrow — EMB=256, Hidden=256, Layers=8

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR |
|-----|---------|---------|-------|-------|-----|
| 0.0005 | 0.4 | 152.12 | 0.207 | 0.414 | 0.307 |
| 0.0005 | 0.5 | 163.70 | 0.199 | 0.404 | 0.299 |
| 0.0005 | 0.6 | 173.48 | 0.188 | 0.394 | 0.288 |

**Notebook 7.2** (1 config): Extreme depth — Hidden=512, Layers=50

| LR | Layers | Val PPL | Notes |
|-----|--------|---------|-------|
| 0.02 | 50 | 672.67 | Catastrophic failure — even after 23 epochs, vanishing/exploding gradients prevented learning |

**Notebook 7.3** (1 config): Hidden=512, Layers=8, Dropout=0.6

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR |
|-----|---------|---------|-------|-------|-----|
| 0.001 | 0.6 | 144.92 | 0.204 | 0.416 | 0.306 |

**Notebook 7.4** (1 config): Hidden=512, Layers=8, Dropout=0.4, 50 epochs

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR | Test PPL |
|-----|---------|---------|-------|-------|-----|----------|
| 0.001 | 0.4 | 130.62 | 0.217 | 0.431 | 0.320 | 164.54 |

**Notebook 7.5** (1 config): Hidden=512, Layers=8, Dropout=0.3, 50 epochs — long training

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR | Test PPL |
|-----|---------|---------|-------|-------|-----|----------|
| 0.0005 | 0.3 | 81.32 | 0.255 | 0.481 | 0.362 | 91.90 |

**Notebook 7.6** (1 config): EMB=256, Hidden=1024, Layers=2, SEQ=50, 50 epochs — 52.1M params

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR |
|-----|---------|---------|-------|-------|-----|
| 0.0005 | 0.4 | 71.35 | 0.263 | 0.496 | 0.373 |

**Notebook 7.7** (1 config): EMB=256, Hidden=2048, Layers=2, SEQ=50

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR | Test PPL |
|-----|---------|---------|-------|-------|-----|----------|
| 0.00075 | 0.4 | 68.61 | 0.268 | 0.503 | 0.379 | 76.32 |

**Notebook 7.8** (1 config): EMB=1024, Hidden=1024, Layers=3, SEQ=70, Batch=128 — 55.9M params

| LR | Dropout | Val PPL | Top-1 | Top-5 | MRR |
|-----|---------|---------|-------|-------|-----|
| 0.001 | 0.4 | **61.81** | 0.278 | 0.515 | 0.389 |

*This is the current notebook's configuration — the best model to date (best at epoch 3).*

### Summary: Best Result per Notebook

| Notebook | Books | EMB | Hidden | Layers | Params | Val PPL | Top-1 | Top-5 | MRR | Test PPL |
|----------|-------|-----|--------|--------|--------|---------|-------|-------|-----|----------|
| 1a | 49 | 256 | 768 | 2 | ~24M | 70.08 | — | — | — | — |
| 1b | 49 | 256 | 1024 | 2 | ~35M | 68.34 | — | — | — | — |
| 2 | 49 | 256 | 2048 | 2 | — | 63.70 | — | — | — | — |
| 3 | 49 | 256 | 2048 | 2 | — | 64.86 | — | — | — | — |
| 4 | 49 | 256 | 2048 | 2 | — | 72.62 | — | — | — | — |
| 5 | 49 | 512 | 4096 | 2 | — | 95.66 | — | — | — | — |
| 6 | 49 | 256 | 2048 | 2 | — | 96.66 | — | — | — | — |
| 7 | 170 | 256 | 1536 | 2 | 77.0M | 101.41 | 0.239 | 0.467 | 0.347 | 127.05 |
| 7.1 | 170 | 256 | 256 | 8 | 17.7M | 152.12 | 0.207 | 0.414 | 0.307 | — |
| 7.2 | 170 | 256 | 512 | 50 | 124.8M | 672.67 | 0.066 | 0.220 | 0.145 | — |
| 7.3 | 170 | 256 | 512 | 8 | 36.5M | 144.92 | 0.204 | 0.416 | 0.306 | — |
| 7.4 | 170 | 256 | 512 | 8 | 36.5M | 130.62 | 0.217 | 0.431 | 0.320 | 164.54 |
| 7.5 | 170 | 256 | 512 | 8 | 39.4M | 81.32 | 0.255 | 0.481 | 0.362 | 91.90 |
| 7.6 | 170 | 256 | 1024 | 2 | 52.1M | 71.35 | 0.263 | 0.496 | 0.373 | — |
| 7.7 | 170 | 256 | 2048 | 2 | — | 68.61 | 0.268 | 0.503 | 0.379 | 76.32 |
| **7.8** | **170** | **1024** | **1024** | **3** | **55.9M** | **61.81** | **0.278** | **0.515** | **0.389** | — |

## 11. Evaluation: Data Volume, Diversity, and Model Size

### 11.1 Data Volume

The experiments span two corpus sizes:

| | Phase 1 (NB 1–6) | Phase 2 (NB 7+) |
|---|---|---|
| Books | 49 | 170 |
| Total tokens | ~3.2 million | ~15.9 million |
| Training tokens | ~2.6M (per-book) or ~2.9M (by-book) | ~12.7 million |
| Validation tokens | ~324K (per-book) or ~320K (by-book) | ~1.6 million |
| Test tokens | ~324K (per-book) or none (by-book) | ~1.6 million |
| Vocabulary size | 30,000 | 30,000 |

This is a **small dataset** by modern language modeling standards — GPT-2 trained on ~10 billion tokens, orders of magnitude more. The limited data makes overfitting a primary concern:
- Without regularization (dropout, weight decay), models quickly memorize training data (train PPL drops to <10 while val PPL stays >100)
- Dropout of 0.4–0.6 is necessary across all successful configurations
- Weight tying provides additional regularization by sharing parameters between embedding and output layers
- The learning rate scheduler (ReduceLROnPlateau) prevents overshooting the narrow optimal region

Expanding from 49 to 170 books roughly 5x'd the data. However, Phase 2 PPL numbers appear higher than Phase 1 for similar architectures. This is partly because Phase 2 uses a per-book split (harder evaluation) and partly because the larger, more diverse corpus is inherently harder to model.

### 11.2 Data Diversity

The corpus is **domain-specific**: fairy tales, folklore, and children's literature from the 18th–early 20th century. This has several implications:

**Advantages**:
- Consistent literary style — the model can learn strong patterns (e.g., "once upon a time", dialogue conventions, narrative structure)
- Manageable vocabulary — fairy tales use relatively simple language, so 30K tokens capture most of the lexicon (low `<unk>` rate)

**Limitations**:
- Narrow domain — the model cannot generalize to other text types (news, code, scientific papers)
- Archaic language — some books use older English ("thee", "thou", "hath") which dilutes the model's focus
- Repetitive motifs — fairy tales share common structures (quest narratives, three-fold repetition, moral endings), which inflates token-level metrics without true "understanding"

The per-book train/val/test split (80/10/10) ensures the model is evaluated on unseen continuations of each book. This is more realistic than random sentence-level splits, which would allow the model to exploit positional patterns within books. The earlier by-book split (NB 1–4) was even stricter — entire books were held out — but lacked a test set.

### 11.3 Model Width (Hidden Size)

Width scaling shows clear diminishing returns (best result per hidden size across all experiments):

| Hidden Size | Best Val PPL | Context | Observation |
|-------------|-------------|---------|-------------|
| 256 | 80–152 | 2–8 layers | Underfitting: insufficient capacity |
| 512 | 73–81 | 2–8 layers | Reasonable baseline, good for deeper models |
| 768 | 70.08 | 2 layers, 49 books | Marginal improvement over 512 |
| 1024 | 61.81 | 3 layers, 170 books, EMB=1024 | Best overall (NB 7.8) |
| 2048 | 63.70 | 2 layers, 49 books | Best on smaller corpus (NB 2) |
| 4096 | 95.66 | 2 layers, 49 books | Overfitting: early stopped after 1 epoch |

The sweet spot is **1024–2048 hidden units**. Below 512, the model lacks capacity. Above 2048, overfitting dominates even with high dropout. The 4096-hidden model (Notebook 5) early-stopped after just 1 epoch — it immediately began overfitting.

### 11.4 Model Depth (Number of Layers)

Depth experiments reveal that LSTMs do **not** benefit from deep stacking on this dataset:

| Layers | Hidden | Best Val PPL | Context | Observation |
|--------|--------|-------------|---------|-------------|
| 2 | 2048 | 63.70 | 49 books | Best on smaller corpus |
| 2 | 2048 | 68.61 | 170 books | Strong baseline |
| 3 | 1024 | **61.81** | 170 books, EMB=1024 | Best overall (NB 7.8) |
| 8 | 512 | 81.32 | 170 books | Requires 48 epochs to converge |
| 8 | 256 | 152.12 | 170 books | Severely underfitting |
| 50 | 512 | 672.67 | 170 books | Near-complete failure after 23 epochs |

**Key finding**: 2–3 layers is optimal. Unlike Transformers, which benefit from dozens of layers due to their attention mechanism, LSTMs suffer from **vanishing gradients** in deep stacks. Even with gradient clipping, information from early layers struggles to propagate through 8+ LSTM layers. The 50-layer experiment (Notebook 7.2) failed to learn meaningful patterns even after 23 epochs.

The 8-layer models can work but require:
- Lower dropout (0.3 vs 0.5–0.6 for 2-layer)
- Much longer training (48 epochs vs 2–5 for 2-layer models)
- Lower learning rates (0.0005)

These models converge to worse final performance despite having more capacity, suggesting the optimization landscape becomes harder with depth.

### 11.5 Embedding Dimension

| EMB | Hidden | Layers | Val PPL | Books | Notes |
|-----|--------|--------|---------|-------|-------|
| 256 | 2048 | 2 | 63.70 | 49 | Small embeddings, large hidden (NB 2) |
| 256 | 1024 | 2 | 68.34 | 49 | Standard configuration (NB 1b) |
| 256 | 2048 | 2 | 68.61 | 170 | Same arch, larger corpus (NB 7.7) |
| 1024 | 1024 | 3 | **61.81** | 170 | Weight-tied, best overall (NB 7.8) |

With weight tying, the embedding dimension equals the projection dimension. Larger embeddings (1024) allow richer token representations and, when combined with 3 LSTM layers, achieve the best metrics. The trade-off is that weight tying constrains the output projection to match the embedding size.


# Checkpoint Loader
Load any saved checkpoint and generate text.

In [22]:
import re

CKPT_PATH = 'checkpoints/best/val_ppl=60.58.pt'


In [24]:
KEEP_PUNCT_AS_TOKENS = True # recommended for generation

SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>", "<num>"]

# Tokenization: words + (optional) punctuation as tokens
# - words: letters/digits/_ and apostrophes inside words
# - punctuation: common sentence punctuation as separate tokens
WORD_RE = r"[a-z]+(?:'[a-z]+)?"
PUNCT_RE = r"[.,!?;:\-—()\[\]\"']"
if KEEP_PUNCT_AS_TOKENS:
    TOKEN_RE = re.compile(rf"{WORD_RE}|{PUNCT_RE}|<num>", re.IGNORECASE)
else:
    TOKEN_RE = re.compile(rf"{WORD_RE}|<num>", re.IGNORECASE)

DIGIT_RE = re.compile(r"\d+")
WHITESPACE_RE = re.compile(r"\s+")

# Optional: normalize curly quotes / dashes to plain forms
CHAR_MAP = str.maketrans({
    "“": '"'   , "”": '"', "„": '"',
    "’": "'", "‘": "'",
    "—": "-", "–": "-",
    "…": "...",
})

def normalize_text(text: str) -> str:
    # normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # unify common unicode punctuation
    text = text.translate(CHAR_MAP)
    # lowercase
    text = text.lower()
    # numbers -> <num>
    text = DIGIT_RE.sub(" <num> ", text)
    # collapse whitespace
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text

def tokenize(text: str) -> List[str]:
    # TOKEN_RE finds tokens directly (words/punct/<num>)
    toks = " ".join(TOKEN_RE.findall(text))
    return toks

## Load checkpoint

In [25]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

hp = ckpt['hyperparams']

print('=== Checkpoint info ===')
print(f"  path:      {CKPT_PATH}")
print(f"  epoch:     {ckpt.get('epoch', '?')}")
print(f"  val_loss:  {ckpt.get('val_loss', '?')}")
print(f"  val_ppl:   {ckpt.get('val_ppl', '?')}")
if 'train_loss' in ckpt:
    print(f"  train_loss:{ckpt['train_loss']}")
if 'global_step' in ckpt:
    print(f"  step:      {ckpt['global_step']}")
print()
print('=== Hyperparams ===')
for k, v in hp.items():
    print(f"  {k}: {v}")

# Build model from stored hyperparams
model = LSTMLM(
    vocab_size=hp['vocab_size'],
    emb_dim=hp['emb_dim'],
    hidden=hp['hidden_size'],
    num_layers=hp['num_layers'],
    dropout=hp['dropout'],
    pad_id=pad_id,
).to(DEVICE)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {total_params:,} params")

=== Checkpoint info ===
  path:      checkpoints/best/val_ppl=60.58.pt
  epoch:     3
  val_loss:  4.103964211279704
  val_ppl:   60.57996399614224
  train_loss:3.72139538277083
  step:      29820

=== Hyperparams ===
  seed: 42
  seq_len: 70
  stride: 10
  batch_size: 128
  grad_clip: 1.0
  emb_dim: 1024
  num_layers: 3
  vocab_size: 30000
  lr: 0.001
  hidden_size: 1024
  dropout: 0.4
  steps_per_epoch: 50000
  max_epochs: 500
  early_stop_patience: 4
  sched_factor: 0.5
  sched_patience: 1
  sched_min_lr: 1e-05
  train_ratio: 0.8
  val_ratio: 0.1
  test_ratio: 0.1

Model loaded: 86,660,400 params


## Sampling utilities

In [26]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    return [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def generate(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8, top_k: int = 50):
    norm = normalize_text(prompt)
    toks = tokenize(norm)
    ids = encode_prompt(toks)
    out = sample_ids(model, ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    text = decode(out)
    print(f'Prompt: {prompt}')
    print(f'---')
    print(text)
    print()
    # ids = encode_prompt(normalize_text(prompt))
    # out = sample_ids(model, ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    # text = decode(out)
    # print(f'Prompt: {prompt}')
    # print(f'---')
    # print(text)
    # print()

print('Ready. Use generate("your prompt here") to sample text.')

Ready. Use generate("your prompt here") to sample text.


## Generate

In [39]:
generate('once upon a time')

Prompt: once upon a time
---
once upon a time, and one night, when she sat on an old woman's head, she cried: " alas! alas! my fortune is mine. give me a cup of milk to drink and drink it with me! " " no, no, " said the wife, " i will make your choice and come; " and the old woman went out into the house. well, the woman went out of the house, and, when she saw the dog, she got up and told her mistress what had happened to him, and said that he had been there for some time with her. then he told her



In [38]:
generate('the princess')

Prompt: the princess
---
the princess, and the princess was so surprised, that the princess asked who she was and why she was called ryn jin. the jelly fish said, " i have been waiting for some time in the emerald palace, but what of course you have got a splendid castle? " " yes, " said the queen, " you will see that the palace contains some great treasure and i have but five hundred and forty of them. " " very well, " answered the princess. " if you wish, " said the sultan, " you must go and ask my daughter. " the soldier said to the king:



In [29]:
generate('in a dark forest')

Prompt: in a dark forest
---
in a dark forest where he had not seen it. i was in a strange country, and my ears were full of the water. i said " yes, " and then he reached his cave. i did not know what he meant, and i made sure the door would be locked, for he was in the same room. i felt that i had been in the house when he was dead. i told him he was going to give me some. a young man had been lying there alone, and it was a long way off, and i could not have told him whether i was alive or dead. " no



In [37]:
generate('once upon a time', max_new_tokens=120, temperature=0.2, top_k=30)

Prompt: once upon a time
---
once upon a time there was a man who had three sons, and they were very poor, and they lived in a great house. one day, when they were out hunting, the old man said to the boy, " go and get some of the fruit which i have brought with you. " the old man was very angry, and said to him, " you have been very kind to me, and you have no other reason to complain of your good fortune. " " i have been thinking of the good fortune of my son, " said the old woman. " i have been thinking of the good fortune of



In [31]:
generate('the princess', max_new_tokens=120, temperature=0.2, top_k=30)

Prompt: the princess
---
the princess, and the king, the queen, and the queen, and the princess, and the princess. the king was delighted to see the princess, and said to him, " you must go and fetch my daughter, and i will give you a present of a diamond ring. " the king was so astonished at the sight, that he ordered his guards to go out and see what was the matter. the princess, who had been so much surprised, said, " i am the princess, and i am the king of the fishes. " " i have been thinking of the princess, " said the



In [36]:
generate('in a dark forest', max_new_tokens=120, temperature=0.2, top_k=30)

Prompt: in a dark forest
---
in a dark forest, and the forest was so deep that it was almost like a great forest. " i have been looking for a great many miles, " said the old man, " and i have heard of a great many things. " " i have been looking for something, " said the old man. " i have been in the forest for a long time. " " i have been thinking of a great many things, " said the old woman. " i have been looking for some place where i can get a good view of the place. " " i have been looking for something, " said the



In [33]:
generate('Now, you must not think of them in this book as learned men, living in great cities, such as they were afterwards, when they wrought all their beautiful works, but as country people, living in farms and walled villages, in a simple, hard-working way; so that the greatest kings and heroes cooked their own meals, and thought it', max_new_tokens=120, temperature=0.5, top_k=30)

Prompt: Now, you must not think of them in this book as learned men, living in great cities, such as they were afterwards, when they wrought all their beautiful works, but as country people, living in farms and walled villages, in a simple, hard-working way; so that the greatest kings and heroes cooked their own meals, and thought it
---
now, you must not think of them in this book as learned men, living in great cities, such as they were afterwards, when they wrought all their beautiful works, but as country people, living in farms and walled villages, in a simple, hard-working way; so that the greatest kings and heroes cooked their own meals, and thought it was a kind of pleasure. but the poor little fellow had a very bad taste; and, besides, he was a great magician, and was only a stupid boy. the king was a great favorite, and lived in a great castle, and had a great many good things, and were all poor; the king was very fond of him, and he used to live in a great palace, and live i

In [35]:
generate('But when the third night came Cinderella was enjoying herself so much that she quite forgot what her Fairy Godmother had said, until suddenly she heard the clock begin to strike twelve. She remembered that as soon as it finished striking, all her fine clothes would turn to rags again; and, jumping up in alarm, she ran out of the room. The Prince ran after her, trying to overtake her; and', max_new_tokens=120, temperature=0.5, top_k=30)

Prompt: But when the third night came Cinderella was enjoying herself so much that she quite forgot what her Fairy Godmother had said, until suddenly she heard the clock begin to strike twelve. She remembered that as soon as it finished striking, all her fine clothes would turn to rags again; and, jumping up in alarm, she ran out of the room. The Prince ran after her, trying to overtake her; and
---
but when the third night came cinderella was enjoying herself so much that she quite forgot what her fairy godmother had said, until suddenly she heard the clock begin to strike twelve. she remembered that as soon as it finished striking, all her fine clothes would turn to rags again; and, jumping up in alarm, she ran out of the room. the prince ran after her, trying to overtake her; and when she got to the door, he opened the door, and ran in, and ran through the bars. but the door was locked, and he could not see the door. the door flew open, and in came the ogre, with a great iron club, 